In [ ]:
# Parameters
run_date = "2026-01-01"  # papermill replacement
import os
output_dir = os.environ.get("ORION_SIGNALS_DIR", "../signals")
dry_run = False

# ensure output exists
os.makedirs(output_dir, exist_ok=True)


In [ ]:
# Import basic modules
import pandas as pd
import numpy as np
import math
import datetime
from datetime import timedelta
from typing import Optional, List, Dict, Any

# Import warnings
import warnings
warnings.filterwarnings("ignore")


In [ ]:
from __future__ import annotations

# -- OPG-STRETCH: the stock is already stretched away from its benchmark before the open -------------
#
# THE TRADE (operator, 2026-11-13, verbatim): "на проміжку 9:15-9:21 знайти ті тікери які стоять із
# відхиленням більше 1 сігма. По колонці ZAP SIG. Вхід на цьому проміжку, вихід о 9:30. Якщо
# відхилення позитивне - ми продаємо, якщо негативне купуємо. Клас лише 1 - 9:30."
#
# So: scan 9:15..9:21 forward; the first minute at which |ZAP SIG| > 1 is the ENTRY, and the position
# is closed at 9:30. Positive reading -> SHORT, negative -> LONG. One class, EXIT0930.
#
# THIS FILE IS OPG-fade WITH THREE CHANGES, and all three matter:
#
#   1. THE SIGNAL AXIS IS `dev_sig`, NOT `deviation`. OPG-fade reads zap% in percentage points;
#      this strategy reads the SAME deviation divided by the ticker's own sigma - final.parquet's
#      own `dev_sig` column, which is what the operator calls ZAP SIG.
#
#      `dev_sig` IS deviation / sigma, exactly. Verified against CRACEN's own
#      `_add_tp_deviation_devsig` and then on 7,336,481 real rows: dev_sig equals
#      round(dev / sigma, 6) in 100.00% of them, max error 0.000e+00.
#
#      THE SUBTLETY, AND THE REASON THE COLUMN IS USED RATHER THAN RECOMPUTED: CRACEN divides
#      BEFORE it rounds. Its order is
#
#          tp  = Stack% * beta ;  dev = tp - Bench% ;  dsg = dev / sigma
#          then  t_p = round(tp, 2) ;  deviation = round(dev, 2) ;  dev_sig = round(dsg, 6)
#
#      so the numerator inside dev_sig is the FULL-PRECISION deviation, while the published
#      `deviation` COLUMN has been rounded to 2 decimal places. Recomputing
#      `deviation / sigma` from the published columns therefore divides an already-rounded
#      numerator and is the LESS exact of the two: measured, it differs from dev_sig by up to
#      0.281 and matches it in only 0.0057% of rows. The gap is exactly the discarded rounding
#      residual over sigma (corr 1.000000; the residual itself never exceeds 0.005, half a 2-dp
#      step), so it is largest where sigma is small - the bottom sigma decile (0.009..0.398)
#      carries gaps up to 0.281, the top decile at most 0.002.
#
#      On a 1-sigma gate the two disagree on 0.109% of rows (8,025 of 7.3M), every one of them
#      with |dev_sig| between 0.9704 and 1.0269 - i.e. only readings already sitting on the
#      threshold. dev_sig is the right side of that disagreement, being the undegraded number.
#
#   2. ENTRY IS INSIDE THE SIGNAL WINDOW, NOT AT THE 9:30 OPEN. Every other OPG file sends the order
#      during 9:20-9:25 and fills at the open, so the signal minute and the entry minute are
#      different things and the signal is read gate-free (the first print in the window, whatever
#      its size). Here the entry IS the triggering minute, so the gate has to move INTO the scan:
#      `_first_trigger_in_window` below walks the window forward and takes the first minute where
#      |ZAP SIG| > the trigger AND a price exists. A ticker whose reading is 0.4 at 9:15 and 2.1 at
#      9:18 enters at 9:18 - under OPG-fade's gate-free read it would have been thrown away on the
#      0.4. Getting this wrong is invisible: the file still runs and still produces trades.
#
#   3. ONE CLASS, EXIT0930 - and it is 9 minutes away at most, not 15-60. The hold is so short that
#      the exit search tolerance is the one thing that could silently invert a trade, so
#      `scan_day` refuses any row whose exit print did not land strictly AFTER its own entry minute.
#
# WHAT STAYS ON Stack%, ON PURPOSE. ZAP SIG decides WHETHER and WHICH WAY to trade. The trade itself
# is an ordinary single-stock position, so entry_val, exit_val and therefore pnl are all read off
# Stack% - the stock's own move is what the position actually banks, and both readings share the same
# settled prev_close, so their difference is the move in percentage points between the two minutes.
# Pricing the pnl in ZAP SIG would be measuring a benchmark-hedged spread this strategy never puts
# on. LAMBDA likewise stays on Stack% (user, 2026-10-11: it must be the identical statistic in every
# family).
#
# ALPHA and GAMMA, by contrast, ARE IN SIGMAS here - they are built from `signals.dev` and
# `trades.entry_dev`, which are now ZAP SIG readings. That is intended (they answer "at what ZAP SIG
# is this ticker worth entering"), but it means GAMMA IS NOT IN PERCENTAGE POINTS in this file, unlike
# every other OPG notebook. The published meta says so explicitly - a consumer that assumes pp would
# be out by a factor of sigma, silently.
#
# MEASURED ON THE REAL FILE BEFORE THIS WAS WRITTEN (three row groups, 78 sessions): in 9:15-9:21,
# 79.7% of prints carry a finite dev_sig, and 61.9% of THOSE already exceed 1 sigma - the median
# |dev_sig| in that window is 1.53. So a 1-sigma trigger is a PERMISSIVE bar on this axis, not a
# selective one (at 9:30 the same figure is 38.3%, because the benchmark-relative stretch is largest
# before the open). ~28 tickers qualify per session, and 2012 of 2016 qualifying (date, ticker) pairs
# had a 9:30 print to exit on. `min_dev_sig` in default_cfg is the knob if that should be tighter.
import math
from collections import namedtuple

import numpy as np
import pandas as pd

SESSION_SPLIT_MIN = 900   # 15:00 - inherited from CLO's own stage split; harmless here since every
                          # OPG-STRETCH checkpoint (9:15..9:30) sits well before it.

Checkpoint = namedtuple("Checkpoint", "name smin tol")

# THE SIGNAL WINDOW IS ALSO THE ENTRY WINDOW (see change 2 in the header). 9:15..9:21 inclusive -
# seven minutes, the operator's own ends, not a centre plus a tolerance.
SIGNAL_FROM_MIN = 9 * 60 + 15   # 9:15
SIGNAL_TO_MIN = 9 * 60 + 21     # 9:21

# LAMBDA is the SAME statistic in every family (user, 2026-10-11): the root-mean-square of the LAST
# Stack% reading before the close - 15:59, searched backwards only up to 15:54 (16:00 and later never
# accepted). On the session axis the prior afternoon's 15:5x prints sit at smin -486..-481.
LAMBDA_SMIN = -481                 # 15:59 on the session axis (15:59 - 24h)
LAMBDA_TOL = 5                     # 15:54 at the furthest
LAMBDA_LO_SMIN = LAMBDA_SMIN - LAMBDA_TOL
LAMBDA_HI_SMIN = LAMBDA_SMIN
CLOSE_SMIN = -480                  # 16:00 - the post-rollover placeholder, never a lambda reading

# ONE exit class. There is no ENTRY checkpoint in this family at all - the entry is whichever minute
# inside the signal window triggered - so this tuple holds the exit alone, and `build_checkpoints`
# has nothing to cap its tolerance against.
SEARCH_WINDOW_MIN = 5
_RAW_CHECKPOINTS = (
    ("EXIT0930", (9, 30), SEARCH_WINDOW_MIN),
)
EXIT_CLASSES = ("EXIT0930",)

# The exit lands the SAME calendar morning as the entry - no midnight crossing anywhere in this
# family, so trade_date == sdate always.
EXIT_DATE_OFFSET = {"EXIT0930": 0}


def _to_smin(hm, split=SESSION_SPLIT_MIN):
    t = hm[0] * 60 + hm[1]
    return t - 1440 if t >= split else t


def build_checkpoints(split=SESSION_SPLIT_MIN):
    """The single EXIT0930 checkpoint. The neighbour-gap cap the other OPG files apply is kept for
    shape parity and is a no-op with one checkpoint; what actually protects this family is the
    exit-after-entry check in `scan_day`, since the entry minute is not fixed here."""
    smins = [_to_smin(hm, split) for _, hm, _ in _RAW_CHECKPOINTS]
    out = {}
    for i, (name, hm, req) in enumerate(_RAW_CHECKPOINTS):
        prev_gap = (smins[i] - smins[i - 1]) if i > 0 else math.inf
        next_gap = (smins[i + 1] - smins[i]) if i + 1 < len(smins) else math.inf
        tol = min(req, prev_gap // 2, next_gap // 2)
        out[name] = Checkpoint(name, smins[i], int(tol))
    return out


def default_cfg(**over):
    cfg = dict(
        min_dev_sig=1.0,      # the trigger, IN SIGMAS: |9:15-9:21 ZAP SIG| must exceed this
                               # (operator: "відхилення більше 1 сігма"). Strict >, so a reading of
                               # exactly 1.0 does not trade.
                               #
                               # NOT comparable to the other OPG files' own `min_dev`, which is in
                               # percentage points. Measured on the real file, 1.0 sigma admits 62%
                               # of the finite readings in this window - it is a loose bar here.
        gamma_floor=1.0,      # ALSO IN SIGMAS, and it starts AT the trigger because nothing below it
                               # exists in the trade log - a lower floor would just re-evaluate the
                               # same full set at every level until it reached 1.0.
        gamma_step=0.1, gamma_max=20.0, gamma_min_n=8, gamma_z=1.96,
                               # gamma_max is 20 rather than the family's usual 15: |dev_sig| reaches
                               # 31 at the 99th percentile in this window, so a 15-sigma cap would be
                               # a real ceiling here rather than the unreachable guard it is on a pp
                               # axis. The walk still stops as soon as evidence runs out.
        max_abs_pp=60.0,      # a Stack% print beyond this is a broken quote, not a price
        max_abs_sig=200.0,    # and a ZAP SIG beyond this is a sigma near zero dividing a real
                               # deviation, not a 200-sigma event. Its own cap because the two
                               # matrices are on completely different scales - cleaning the sigma
                               # axis at 60 would throw away genuine 60+ sigma readings, of which
                               # this window has plenty.
    )
    cfg.update(over)
    return cfg


def panel_from_stack(tickers, t_first, stack, zsig=None, prev_close=None):
    """One value per ticker per minute - Stack% itself, plus `zsig` (final.parquet's own `dev_sig`,
    the benchmark-relative move in sigmas) on the SAME grid. `t_first` is the SMIN of column 0.

    Both matrices are carried because the two are used for different jobs and must not be confused:
    zsig decides the signal and the direction, stack prices the trade. A panel built without `zsig`
    still works for lambda, and scan_day says so loudly rather than falling back to Stack%."""
    out = dict(tickers=np.asarray(tickers), t_first=int(t_first), stack=np.asarray(stack, np.float32))
    if zsig is not None:
        out["zsig"] = np.asarray(zsig, np.float32)
    if prev_close is not None:
        out["prev_close"] = np.asarray(prev_close, np.float32)
    return out


def _clean(mat, cap):
    """Drop non-finite values and anything past `cap` in absolute value. `cap` is passed in rather
    than read from cfg because the two matrices have different ones - see max_abs_sig."""
    bad = ~np.isfinite(mat) | (np.abs(mat) > cap)
    out = mat.copy()
    out[bad] = np.nan
    return out


def _first_in_window(mat, t_first, lo_min, hi_min):
    """The GATE-FREE read: scan lo_min..hi_min forward and take the first finite print per ticker,
    whatever its size. This is ALPHA's population - every reading the ticker produced, traded or
    not - and it is deliberately NOT what the entry uses here. Sentinel minute -1."""
    N, T = mat.shape
    val = np.full(N, np.nan, np.float32)
    used = np.full(N, -1, np.int64)
    remaining = np.arange(N)
    lo_j = max(0, lo_min - t_first)
    hi_j = min(T - 1, hi_min - t_first)
    for j in range(lo_j, hi_j + 1):
        if remaining.size == 0:
            break
        v = mat[remaining, j]
        hit = np.isfinite(v)
        if hit.any():
            idx = remaining[hit]
            val[idx] = v[hit]
            used[idx] = j
            remaining = remaining[~hit]
    minute = np.where(used >= 0, used + t_first, -1)
    return val, minute


def _first_trigger_in_window(sig_mat, price_mat, t_first, lo_min, hi_min, thr):
    """THE ENTRY RULE, and the one piece of machinery this family needs that no other OPG file has.

    Scan lo_min..hi_min FORWARD and take, per ticker, the first minute that satisfies ALL of:
      * a finite ZAP SIG reading,
      * |reading| strictly greater than `thr`,
      * AND a finite Stack% print in the SAME minute.

    The last condition is not a technicality: the entry is filled at that minute, so a minute with a
    signal and no price is not an entry, and silently carrying the signal forward to the next priced
    minute would record a fill the strategy never had.

    -> (signal value, entry price reading, minute). Minute is the sentinel -1 for a ticker that never
    triggered - which is the overwhelming majority of them, by design."""
    N, T = sig_mat.shape
    sig = np.full(N, np.nan, np.float32)
    px = np.full(N, np.nan, np.float32)
    used = np.full(N, -1, np.int64)
    remaining = np.arange(N)
    lo_j = max(0, lo_min - t_first)
    hi_j = min(T - 1, hi_min - t_first)
    for j in range(lo_j, hi_j + 1):
        if remaining.size == 0:
            break
        s = sig_mat[remaining, j]
        p = price_mat[remaining, j]
        hit = np.isfinite(s) & np.isfinite(p) & (np.abs(s) > thr)
        if hit.any():
            idx = remaining[hit]
            sig[idx] = s[hit]
            px[idx] = p[hit]
            used[idx] = j
            remaining = remaining[~hit]
    minute = np.where(used >= 0, used + t_first, -1)
    return sig, px, minute


def _nearest(mat, t_first, t_col, tol, exclude=None, prefer_later=False):
    """value + the absolute minute actually used, per row, searching outward from t_col: 0, -1, +1,
    -2, +2, ... (or 0, +1, -1, +2, -2, ... when `prefer_later=True`) so a tie between an earlier and a
    later print keeps the preferred side - the first ring that finds a print wins, and a row already
    filled is never looked at again.

    `prefer_later`: the EXIT class uses True, matching the live engines' own preferLaterOnTie=true for
    exits. There is no ENTRY call site in this family.

    `exclude`: raw minute indices never accepted even if finite - used only by the lambda read.

    Sentinel: a row with NO match anywhere in the window gets `minute == -1` (the literal integer).
    """
    N, T = mat.shape
    val = np.full(N, np.nan, np.float32)
    used = np.full(N, -1, np.int64)
    remaining = np.arange(N)
    j0 = t_col - t_first
    excl_j = {m - t_first for m in exclude} if exclude else set()
    for d in range(0, int(tol) + 1):
        if remaining.size == 0:
            break
        order = (j0,) if d == 0 else ((j0 + d, j0 - d) if prefer_later else (j0 - d, j0 + d))
        for j in order:
            if remaining.size == 0:
                break
            if j < 0 or j >= T or j in excl_j:
                continue
            v = mat[remaining, j]
            hit = np.isfinite(v)
            if hit.any():
                idx = remaining[hit]
                val[idx] = v[hit]
                used[idx] = j
                remaining = remaining[~hit]
    minute = np.where(used >= 0, used + t_first, -1)
    return val, minute


def scan_day(panel, cfg, checkpoints):
    """-> (trades, signals, entries) for one session (one calendar morning - no midnight crossing).

    trades:  one row per ticker that triggered inside 9:15-9:21 and had a 9:30 exit print strictly
             after its own entry minute. One class only (EXIT0930).
    signals: the first finite 9:15-9:21 ZAP SIG reading for every ticker that had one, gated on
             nothing - ALPHA's source, in SIGMAS.
    entries: the last pre-close (15:59, backwards to 15:54) Stack% reading for every ticker that had
             one, gated on nothing - LAMBDA's source, identical to every other family. Note that the
             name is inherited and is NOT this strategy's entry price: the traded entry is a 9:15-9:21
             print and lives in trades.entry_val."""
    stack = _clean(panel["stack"], cfg["max_abs_pp"])
    t_first = panel["t_first"]
    tk = panel["tickers"]

    # No silent fallback: a panel staged by another OPG notebook has no zsig matrix, and running it
    # here would quietly reproduce that other strategy under this file's name.
    if "zsig" not in panel:
        raise KeyError("panel has no 'zsig' matrix - stage with opgstretch_stage_by_session, "
                       "not another OPG stage")
    zsig = _clean(panel["zsig"], cfg["max_abs_sig"])

    # ALPHA's gate-free population, and the ENTRY, read off the same window by two different rules -
    # see _first_trigger_in_window's own docstring for why the entry cannot reuse the gate-free one.
    sig_any, _ = _first_in_window(zsig, t_first, SIGNAL_FROM_MIN, SIGNAL_TO_MIN)
    sig_v, ent_v, ent_m = _first_trigger_in_window(
        zsig, stack, t_first, SIGNAL_FROM_MIN, SIGNAL_TO_MIN, cfg["min_dev_sig"])

    # LAMBDA's population: the last pre-close reading, from the previous afternoon's slice staged
    # into this session.
    lam_v, _ = _nearest(stack, t_first, LAMBDA_SMIN, LAMBDA_TOL,
                        exclude=set(range(CLOSE_SMIN, LAMBDA_SMIN + LAMBDA_TOL + 1)), prefer_later=False)
    lam_ok = np.isfinite(lam_v)
    entries = pd.DataFrame({"ticker": tk[lam_ok], "entry_raw": lam_v[lam_ok]})

    any_ok = np.isfinite(sig_any)
    signals = pd.DataFrame({"ticker": tk[any_ok], "dev": sig_any[any_ok]})

    have = ent_m >= 0
    if not have.any():
        return _EMPTY_TRADES.copy(), signals, entries

    idx0 = np.flatnonzero(have)

    rows = []
    for cls in EXIT_CLASSES:
        ck = checkpoints[cls]
        ex_v, ex_m = _nearest(stack, t_first, ck.smin, ck.tol, prefer_later=True)
        # THE EXIT MUST LAND AFTER THE ENTRY. With a 9-minute hold and a 5-minute exit search the
        # two windows do not currently overlap (the earliest exit candidate is 9:25, the latest entry
        # 9:21), so this drops nothing today - it is here because that is a property of two
        # constants, and widening either one would otherwise start recording trades held for a
        # negative number of minutes, with a pnl of the right magnitude and the wrong sign.
        ex_ok = np.isfinite(ex_v[idx0]) & (ex_m[idx0] > ent_m[idx0])
        if not ex_ok.any():
            continue
        j = idx0[ex_ok]
        e_dev = sig_v[j]
        e_val = ent_v[j]
        x_val = ex_v[j]
        # Direction: positive ZAP SIG -> Short, negative -> Long (operator). x_val/e_val are Stack%,
        # so this is what the single-stock position actually banks over the hold.
        pnl = -np.sign(e_dev) * (x_val - e_val)
        rows.append(pd.DataFrame({
            "ticker": tk[j], "cls": cls, "sign": np.where(e_dev > 0, "pos", "neg"),
            "entry_dev": e_dev, "entry_val": e_val, "exit_val": x_val, "pnl": pnl.astype(np.float32),
            # signal_minute == entry_minute in this family, by construction: the triggering minute IS
            # the fill. Both are written so the published row shape stays identical to the other OPG
            # files, whose two minutes genuinely differ.
            "signal_minute": ent_m[j], "entry_minute": ent_m[j], "exit_minute": ex_m[j],
            "hold_minutes": (ex_m[j] - ent_m[j]).astype(np.int64),
            "date_offset": EXIT_DATE_OFFSET[cls],
        }))
    trades = pd.concat(rows, ignore_index=True) if rows else _EMPTY_TRADES.copy()
    return trades, signals, entries


_EMPTY_TRADES = pd.DataFrame(columns=["ticker", "cls", "sign", "entry_dev", "entry_val", "exit_val", "pnl",
                                       "signal_minute", "entry_minute", "exit_minute", "hold_minutes",
                                       "date_offset"])


def _add_days(sdate_int, offset_days):
    """int YYYYMMDD + a signed day offset -> int YYYYMMDD. Always a no-op in this family
    (EXIT_DATE_OFFSET is all zeros) - kept for shape parity with the other scan_days."""
    if offset_days == 0:
        return sdate_int
    ts = pd.Timestamp(str(sdate_int)) + pd.Timedelta(days=int(offset_days))
    return int(ts.strftime("%Y%m%d"))


def scan_days(day_iter, cfg, checkpoints=None):
    checkpoints = checkpoints or build_checkpoints()
    tr_all, sg_all, en_all = [], [], []
    for sdate, panel in day_iter:
        tr, sg, en = scan_day(panel, cfg, checkpoints)
        if len(tr):
            tr = tr.copy()
            tr.insert(0, "sdate", int(sdate))
            tr["trade_date"] = [_add_days(int(sdate), off) for off in tr["date_offset"]]
            tr_all.append(tr.drop(columns="date_offset"))
        if len(sg):
            sg = sg.copy()
            sg.insert(0, "sdate", int(sdate))
            sg_all.append(sg)
        if len(en):
            en = en.copy()
            en.insert(0, "sdate", int(sdate))
            en_all.append(en)
    trades = pd.concat(tr_all, ignore_index=True) if tr_all else (
        _EMPTY_TRADES.copy().assign(sdate=pd.Series(dtype=int), trade_date=pd.Series(dtype=int))
        [["sdate", *[c for c in _EMPTY_TRADES.columns if c != "date_offset"], "trade_date"]])
    signals = pd.concat(sg_all, ignore_index=True) if sg_all else pd.DataFrame(columns=["sdate", "ticker", "dev"])
    entries = pd.concat(en_all, ignore_index=True) if en_all else pd.DataFrame(columns=["sdate", "ticker", "entry_raw"])
    return trades, signals, entries


In [ ]:
from __future__ import annotations

# -- ratings, alpha, gamma, lambda from the trade/signal/entries tables scan_days produces -----------
# Identical machinery to every other family's own ratings cell - alpha/gamma/lambda are computed
# purely from the trades/signals/entries tables, which carry no notion of the thesis themselves.
#
# OPG-STRETCH note, and it is the one thing to carry out of this cell: ALPHA and GAMMA ARE IN SIGMAS
# HERE, not in percentage points. They read `signals.dev` and `trades.entry_dev`, which this family
# fills with ZAP SIG readings, so a "gamma of 2.4" means 2.4 sigma, not 2.4 pp. LAMBDA is still
# Stack% in pp, exactly as in every other family (user, 2026-10-11: it must be the identical
# statistic everywhere). Mixing the two units is the one silent failure available in this file.
import math

import numpy as np
import pandas as pd


def _wilson_lb(k, n, z=1.96):
    k = np.asarray(k, float)
    n = np.asarray(n, float)
    with np.errstate(all="ignore"):
        p = k / n
        den = 1 + z * z / n
        c = p + z * z / (2 * n)
        r = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
        return np.where(n > 0, (c - r) / den, np.nan)


def _modal_value(values, step=0.1, min_n=3):
    """The value a stack gives MOST OFTEN, i.e. the mode of the reading, not its RMS. |values| are
    binned at the same granularity GAMMA walks in (gamma_step), the modal bin wins, and the reported
    value is the MEAN of the raw readings that landed in it. NaN below min_n readings."""
    v = np.asarray(values, float)
    v = v[np.isfinite(v)]
    if v.size < min_n:
        return float("nan")
    bins = np.round(v / step) * step
    uniq, counts = np.unique(bins, return_counts=True)
    best = uniq[np.argmax(counts)]
    in_bin = np.abs(bins - best) < 1e-9
    return float(np.mean(v[in_bin]))


def compute_alpha(signals, step=0.1, min_n=3):
    """ALPHA, per ticker, split pos/neg: the modal |9:15-9:21 ZAP SIG reading| on each side, over
    EVERY reading the ticker produced - whether or not it was big enough to trade, the same gate-free
    universe every other family uses. IN SIGMAS.
    -> DataFrame indexed by ticker, columns alpha_pos/alpha_neg."""
    if not len(signals):
        return pd.DataFrame(columns=["alpha_pos", "alpha_neg"]).rename_axis("ticker")
    out = {}
    for tk, g in signals.groupby("ticker"):
        dev = g["dev"].to_numpy()
        out[tk] = (
            _modal_value(dev[dev > 0], step, min_n),
            _modal_value(-dev[dev < 0], step, min_n),
        )
    return pd.DataFrame.from_dict(out, orient="index", columns=["alpha_pos", "alpha_neg"]).rename_axis("ticker")


def compute_lambda(entries, min_n=2):
    """LAMBDA, per ticker: root-mean-square (sqrt(mean(x^2)), NOT std) of the last pre-close Stack%
    reading (15:59, searched backwards up to 15:54; yesterday's close as reference = the day's
    close-to-close move), IN PERCENTAGE POINTS. Identical to every other family - see scan_day.
    Gate-free, same universe as ALPHA. NaN below min_n readings.
    -> Series indexed by ticker, named "lambda"."""
    if not len(entries):
        return pd.Series(dtype=float, name="lambda").rename_axis("ticker")
    counts = entries.groupby("ticker")["entry_raw"].transform("count")
    # root-mean-square (NOT std): no mean subtraction, no ddof - sqrt(mean(r^2))
    out = (entries[counts >= min_n].assign(_sq=lambda d: d["entry_raw"].astype(float) ** 2)
           .groupby("ticker")["_sq"].mean().pow(0.5))
    return out.rename("lambda").rename_axis("ticker")


def compute_ratings(trades):
    """total/wins/win_rate(+Wilson LB)/mean pnl(+95% LB)/median, per (ticker, cls, sign). With one
    exit class in this family, `cls` is always EXIT0930 - the column is kept because every consumer
    of these files keys on it."""
    if not len(trades):
        return pd.DataFrame(columns=["ticker", "cls", "sign", "total", "wins", "win_rate", "win_rate_lb",
                                     "mean_pnl", "median_pnl", "pnl_lb"])
    g = trades.groupby(["ticker", "cls", "sign"])
    out = g["pnl"].agg(total="size", mean_pnl="mean", median_pnl="median", sd_pnl="std").reset_index()
    out["wins"] = g["pnl"].apply(lambda s: int((s > 0).sum())).to_numpy()
    out["win_rate"] = out["wins"] / out["total"]
    out["win_rate_lb"] = _wilson_lb(out["wins"], out["total"])
    with np.errstate(all="ignore"):
        se = out["sd_pnl"] / np.sqrt(out["total"])
        out["pnl_lb"] = np.where(out["total"] > 1, out["mean_pnl"] - 1.96 * se, np.nan)
    return out.drop(columns="sd_pnl")


def compute_gamma(trades, cfg):
    """GAMMA, per (ticker, cls, sign): the LOWEST level L >= gamma_floor at which trades reaching
    |entry_dev| >= L have a positive 95% lower bound on mean pnl - "the lowest reading at which this
    ticker has actually paid".

    IN SIGMAS in this file (entry_dev is a ZAP SIG reading), and the floor is the trigger itself:
    nothing below 1 sigma is in the trade log at all, so starting lower would only re-test the same
    full set repeatedly."""
    if not len(trades):
        return pd.DataFrame(columns=["ticker", "cls", "sign", "gamma", "gamma_n", "gamma_mean", "gamma_rate"])
    rows = []
    for (tk, cls, sg), g in trades.groupby(["ticker", "cls", "sign"]):
        dev = np.abs(g["entry_dev"].to_numpy())
        pnl = g["pnl"].to_numpy()
        lvl = float(cfg["gamma_floor"])
        found = None
        while lvl <= cfg["gamma_max"] + 1e-9:
            m = dev >= lvl - 1e-9
            n = int(m.sum())
            if n < cfg["gamma_min_n"]:
                break
            mp = float(pnl[m].mean())
            se = float(pnl[m].std(ddof=1)) / math.sqrt(n) if n > 1 else math.inf
            lb = mp - cfg["gamma_z"] * se
            if lb > 0:
                found = (lvl, n, mp, float((pnl[m] > 0).mean()))
                break
            lvl = round(lvl + cfg["gamma_step"], 6)
        if found is not None:
            rows.append({"ticker": tk, "cls": cls, "sign": sg, "gamma": found[0], "gamma_n": found[1],
                        "gamma_mean": found[2], "gamma_rate": found[3]})
    return pd.DataFrame(rows, columns=["ticker", "cls", "sign", "gamma", "gamma_n", "gamma_mean", "gamma_rate"])


def build_summary(trades, signals, cfg, sigma_by_ticker=None, lambda_by_ticker=None, static_by_ticker=None):
    """alpha + ratings + gamma, joined into one per-(ticker, cls, sign) table. "pos"/"neg" label the
    DIRECTION OF THE 9:15-9:21 ZAP SIG reading, not the trade taken - identical labelling to every
    other family even though a "pos" row here is a SHORT (sell the stretched-up stock). `sigma` is
    read from the same final.parquet column every strategy reads (a ticker property, not
    strategy-specific) - and here it is also the denominator the whole signal axis is expressed in,
    so it is worth having beside the levels rather than only as a descriptor. `lambda` is
    `compute_lambda`'s own per-ticker Series, computed fresh each run from this run's own entries
    table."""
    alpha = compute_alpha(signals)
    ratings = compute_ratings(trades)
    gamma = compute_gamma(trades, cfg)
    out = ratings.merge(gamma, on=["ticker", "cls", "sign"], how="left")

    alpha_pos = out["ticker"].map(alpha["alpha_pos"]) if len(alpha) else np.nan
    alpha_neg = out["ticker"].map(alpha["alpha_neg"]) if len(alpha) else np.nan
    out["alpha"] = np.where(out["sign"] == "pos", alpha_pos, alpha_neg)

    if sigma_by_ticker:
        out["sigma"] = out["ticker"].map(sigma_by_ticker)
    else:
        out["sigma"] = np.nan
    # the same three identification fields the scout row carries, so this strategy's own
    # <code>_ratings.csv is self-contained for the Scanner/Stream/Sonar rho/beta filters and does
    # not need a second lookup into the rolling_perf file.
    _stat = static_by_ticker or {}
    for _f in ("bench", "corr", "beta"):
        out[_f] = [(_stat.get(t) or {}).get(_f) for t in out["ticker"]]

    if lambda_by_ticker is not None and len(lambda_by_ticker):
        out["lambda"] = out["ticker"].map(lambda_by_ticker)
    else:
        out["lambda"] = np.nan

    return out, alpha


In [ ]:
from __future__ import annotations

# -- stage: final.parquet -> one small parquet per SESSION --------------------------------------------
# OPG-STRETCH stages Stack% alongside `dev_sig` - final.parquet's own ZAP SIG column. Both are needed
# on the same grid (dev_sig for the signal and the direction, Stack% for entry/exit/pnl/lambda), so
# they are staged together rather than joined later, and the stage dir is its own
# (opgstretch_stage_v2) so a panel can never be read by the wrong notebook.
#
# WHY `dev_sig` AND NOT `deviation` / `sigma`: the operator reads ZAP SIG, and `dev_sig` IS that
# column - deviation over sigma exactly (100.00% of 7,336,481 real rows, max error 0.000e+00
# against CRACEN's own formula). What it is NOT is `published deviation / sigma`: CRACEN divides
# before it rounds, so dev_sig holds the full-precision numerator while the `deviation` column has
# been rounded to 2 dp. Recomputing from the published columns is the degraded version - up to 0.281
# off, and worst exactly where sigma is small. See the core cell's header for the full measurement.
# The per-ticker `sigma` is ALSO collected, but as a descriptor for the ratings table, never as a
# divisor.
#
# prev_close is NOT staged (unlike CLO): this family never rebases anything - the entry and the exit
# sit minutes apart on the same morning against the same already-settled reference close.
import numpy as np
import pandas as pd

# SESSION_SPLIT_MIN, LAMBDA_* and panel_from_stack come from this notebook's own core cell, executed
# into the same namespace before this cell (the notebook order) - not imported.


def opgstretch_stage_by_session(input_path, stage_dir, *, session_split_min=SESSION_SPLIT_MIN,
                        start_date=None,
                        STOCK_NUM_FIELD="Stack%", ZSIG_NUM_FIELD="dev_sig", SIGMA_FIELD="sigma",
                        BENCH_FIELD="bench", CORR_FIELD="corr", BETA_FIELD="beta",
                        smin_lo=520, smin_hi=650, log_every_n_chunks=20):
    """-> (sorted session dates, {ticker: static sigma}).

    Sigma is READ, not computed: the source final.parquet already carries a per-row "sigma" column,
    the same static-per-ticker dispersion figure every strategy reads. The LATEST non-null value seen
    per ticker wins (row groups are read in file order, which is time order).

    smin_lo/smin_hi default to 520/650 (8:40..10:50), which covers this family's own 9:15..9:30
    window with room on both sides; the lambda slice is pulled in separately by the `keep` mask
    below, exactly as in the other OPG stages."""
    import gc, time, shutil
    import pyarrow as pa
    import pyarrow.parquet as pq
    from pathlib import Path

    stage = Path(stage_dir)
    if stage.exists():
        shutil.rmtree(stage)
    stage.mkdir(parents=True, exist_ok=True)
    start_i = int(start_date.replace("-", "")) if start_date else -1

    schema = pa.schema([("ticker", pa.string()), ("smin", pa.int16()), ("stack", pa.float32()),
                        ("zsig", pa.float32())])
    writers, counts = {}, {}
    sigma_by_ticker: dict = {}
    static_by_ticker: dict = {}

    def _writer(sdate):
        if sdate not in writers:
            writers[sdate] = pq.ParquetWriter(str(stage / f"{sdate}.parquet"), schema, compression="zstd")
            counts[sdate] = 0
        return writers[sdate]

    pf = pq.ParquetFile(input_path)
    # ZSIG_NUM_FIELD is REQUIRED, not optional: without it this strategy has no signal at all. The
    # sibling OPG-fade file records that this exact column was once left out of the `want` list while
    # the reader still guarded with "if in df.columns", which staged a column of pure NaN and
    # produced a notebook that ran clean and never traded.
    want = ["ticker", "dt", STOCK_NUM_FIELD, ZSIG_NUM_FIELD]
    cols = [c for c in want if c in pf.schema.names]
    for need in want:
        if need not in cols:
            raise KeyError(f"final.parquet is missing required column: {need}")
    has_sigma = SIGMA_FIELD in pf.schema.names
    if has_sigma:
        cols = cols + [SIGMA_FIELD]
    else:
        print(f"WARN: final.parquet has no '{SIGMA_FIELD}' column - sigma will be null for every ticker")
    # -- SPECTR / Scout reference block -------------------------------------------------------------
    # bench/corr/beta join sigma as the ticker's IDENTIFICATION row. All four are per-ticker CONSTANTS
    # in final.parquet (checked on the rebuilt file: exactly one distinct value per ticker each), so
    # they are harvested by the rule sigma always used - latest non-null wins - and published on the
    # scout row. A column the file does not carry stays null for every ticker; it is never defaulted,
    # the house rule being that a row missing the field a filter reads fails that filter.
    STATIC_FIELDS = [f for f in (SIGMA_FIELD, BENCH_FIELD, CORR_FIELD, BETA_FIELD) if f in pf.schema.names]
    for _missing in (f for f in (BENCH_FIELD, CORR_FIELD, BETA_FIELD) if f not in pf.schema.names):
        print(f"WARN: final.parquet has no '{_missing}' column - it will be null for every ticker")
    cols = cols + [f for f in STATIC_FIELDS if f not in cols]
    t0 = time.time()
    total_in = total_out = 0
    print(f"START opgstretch stage  file={input_path}  session_split={session_split_min}min")
    try:
        for ci in range(pf.num_row_groups):
            df = pf.read_row_group(ci, columns=cols).to_pandas()
            total_in += len(df)
            dt = pd.to_datetime(df["dt"], errors="coerce", utc=True)
            ok = dt.notna().to_numpy(copy=False)
            if not ok.any():
                continue
            dt, df = dt[ok], df.loc[ok]
            t_arr = dt.dt.hour.to_numpy(dtype="int32", copy=False) * 60 + dt.dt.minute.to_numpy(dtype="int32", copy=False)
            late = t_arr >= session_split_min
            smin = np.where(late, t_arr - 1440, t_arr).astype("int16")
            sess = dt + pd.to_timedelta(np.where(late, 1, 0), unit="D")
            sdate = (sess.dt.year.to_numpy(dtype="int32", copy=False) * 10000 + sess.dt.month.to_numpy(dtype="int32", copy=False) * 100
                     + sess.dt.day.to_numpy(dtype="int32", copy=False)).astype("int32")
            stack = pd.to_numeric(df[STOCK_NUM_FIELD], errors="coerce").to_numpy(dtype="float32", copy=False)
            zsig = pd.to_numeric(df[ZSIG_NUM_FIELD], errors="coerce").to_numpy(dtype="float32", copy=False)
            # The four static fields, harvested together. VECTORISED on purpose: the loop this
            # replaces ran one Python iteration per TAPE ROW (~1.0B rows on the current file); this
            # one runs over each row group's distinct tickers, with the same "latest non-null wins".
            for _f in STATIC_FIELDS:
                _sub = df.loc[df[_f].notna(), ["ticker", _f]]
                if not len(_sub):
                    continue
                _sub = _sub.drop_duplicates("ticker", keep="last")
                for _tk, _v in zip(_sub["ticker"].to_numpy(copy=False), _sub[_f].to_numpy(copy=False)):
                    static_by_ticker.setdefault(str(_tk), {})[_f] = (
                        str(_v) if _f == BENCH_FIELD else float(_v))
            # Still gated on a finite Stack% ONLY: lambda and every price need it, while a row whose
            # dev_sig is null (no Bench% print that minute) is a row this strategy cannot signal on -
            # it must not drag the price series out with it. Measured on the real file, about 20% of
            # the prints in the 9:15-9:21 window are exactly that case.
            keep = (((smin >= smin_lo) & (smin <= smin_hi)) | ((smin >= LAMBDA_LO_SMIN) & (smin <= LAMBDA_HI_SMIN))) & np.isfinite(stack)
            if start_i > 0:
                keep &= sdate >= start_i
            if not keep.any():
                continue
            out = pd.DataFrame({"ticker": df["ticker"].to_numpy(copy=False)[keep].astype(str), "sdate": sdate[keep],
                                "smin": smin[keep], "stack": stack[keep], "zsig": zsig[keep]})
            for sd, part in out.groupby("sdate", sort=False):
                _writer(int(sd)).write_table(pa.Table.from_pandas(part[["ticker", "smin", "stack", "zsig"]], schema=schema, preserve_index=False))
                counts[int(sd)] += len(part)
                total_out += len(part)
            del df, out
            if (ci + 1) % log_every_n_chunks == 0:
                print(f"[rg {ci + 1:>4}/{pf.num_row_groups}] in={total_in:,} staged={total_out:,} sessions={len(writers)} elapsed={time.time() - t0:.1f}s")
                gc.collect()
    finally:
        for w in writers.values():
            w.close()
    # sigma stays its own flat map so every existing caller keeps working unchanged; the full
    # block travels beside it for the scout / SPECTR row.
    if has_sigma:
        sigma_by_ticker = {t: d[SIGMA_FIELD] for t, d in static_by_ticker.items() if SIGMA_FIELD in d}
    print(f"DONE stage in={total_in:,} staged={total_out:,} sessions={len(writers)} "
          f"sigma_tickers={len(sigma_by_ticker):,} elapsed={time.time() - t0:.1f}s")
    return sorted(counts), sigma_by_ticker, static_by_ticker


def opgstretch_sessions(stage_dir, *, smin_lo=520, smin_hi=650, min_tickers=30):
    """yields (sdate, panel) in date order, one pivoted session at a time."""
    from pathlib import Path
    for f in sorted(Path(stage_dir).glob("*.parquet"), key=lambda p: int(p.stem)):
        df = pd.read_parquet(f)
        if df["ticker"].nunique() < min_tickers:
            continue
        df = df.drop_duplicates(["ticker", "smin"], keep="last")
        tick, ti = np.unique(df["ticker"].to_numpy(), return_inverse=True)
        lo = min(smin_lo, LAMBDA_LO_SMIN)
        T = smin_hi - lo + 1
        stack = np.full((tick.size, T), np.nan, np.float32)
        zsig = np.full((tick.size, T), np.nan, np.float32)
        col = df["smin"].to_numpy().astype(np.int64) - lo
        stack[ti, col] = df["stack"].to_numpy()
        zsig[ti, col] = df["zsig"].to_numpy()
        yield int(f.stem), panel_from_stack(tick, lo, stack, zsig=zsig)


In [ ]:
from pathlib import Path
import os, gzip, json, datetime


def _resolve_orion_paths(strategy_code: str):
    final_env = os.environ.get("FINAL_PARQUET_PATH")
    sig_env = os.environ.get("SIGNALS_DIR")
    orion_env = os.environ.get("ORION_HOME")

    final_path = Path(final_env).expanduser().resolve() if final_env else None
    signals_base = Path(sig_env).expanduser().resolve() if sig_env else None

    if (final_path is None or signals_base is None) and orion_env:
        orion_home = Path(orion_env).expanduser().resolve()
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    if final_path is None or signals_base is None:
        here = Path.cwd().resolve()
        orion_home = None
        for parent in [here] + list(here.parents):
            if parent.name.lower() == "orion":
                orion_home = parent
                break
            cand = parent / "OriON"
            if cand.exists() and cand.is_dir():
                orion_home = cand.resolve()
                break
        if orion_home is None:
            raise RuntimeError("Cannot locate OriON. Set ORION_HOME env var (recommended).")
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    out_dir = (signals_base / strategy_code.lower()).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    if not final_path.exists():
        raise FileNotFoundError(f"FINAL parquet not found: {final_path}")
    return final_path, out_dir


# -- runner -------------------------------------------------------------------------------------------
# Strategy code "opgstretch" - its own folder and its own files. No bridge service reads these yet: an
# OPGStretch ratings service / rolling-perf parser would be a near-copy of the OPGReversal pair, since
# the wire shape here is identical ("cls" UPPERCASE, "sign" pos/neg, same columns) - with ONE caveat
# that a port must not miss: this family's alpha/gamma are in SIGMAS, not percentage points, and its
# entry is a 9:15-9:21 print rather than the 9:30 open.
FINAL_PATH, OUT_DIR = _resolve_orion_paths("opgstretch")

STAGE_DIR = OUT_DIR.parent.parent / "_work" / "opgstretch_stage_v2"
STAGE_DIR.parent.mkdir(parents=True, exist_ok=True)
# The sidecar caches the WHOLE static block (bench/corr/beta/sigma), not sigma alone, so a
# REUSED stage still yields a complete SPECTR row. Deliberately a new filename: an old
# sigma-only sidecar must not be mistaken for this one, which would silently republish null
# bench/corr/beta and read as a data problem rather than a stale cache.
STATIC_SIDECAR = STAGE_DIR.parent / "opgstretch_stage_static.json"

_stage_ok = STAGE_DIR.exists() and any(STAGE_DIR.glob("*.parquet")) and STATIC_SIDECAR.exists() and \
    min(p.stat().st_mtime for p in STAGE_DIR.glob("*.parquet")) >= FINAL_PATH.stat().st_mtime
if _stage_ok:
    print("stage is at least as new as final.parquet (incl. its static sidecar) - reusing", STAGE_DIR)
    static_by_ticker = json.loads(STATIC_SIDECAR.read_text(encoding="utf-8"))
    sigma_by_ticker = {t: d["sigma"] for t, d in static_by_ticker.items() if d.get("sigma") is not None}
else:
    _, sigma_by_ticker, static_by_ticker = opgstretch_stage_by_session(str(FINAL_PATH), str(STAGE_DIR))
    STATIC_SIDECAR.write_text(json.dumps(static_by_ticker), encoding="utf-8")
    if not sigma_by_ticker:
        print("WARN: fresh stage still produced no sigma for any ticker - check final.parquet's own "
              "'sigma' column")

CFG = default_cfg()
CHECKPOINTS = build_checkpoints()

trades, signals, entries = scan_days(opgstretch_sessions(str(STAGE_DIR)), CFG, CHECKPOINTS)
print(f"trades: {len(trades):,} over {trades['sdate'].nunique() if len(trades) else 0} sessions, "
      f"{trades['ticker'].nunique() if len(trades) else 0} tickers")
if len(trades):
    # The two numbers worth seeing on every run. The hold is 9 minutes at most by construction, so a
    # median far from that means the 9:30 print is routinely being found late; and a trigger that
    # admits nearly every reading is a trigger that is not selecting anything.
    print(f"hold minutes: min {int(trades['hold_minutes'].min())} / median "
          f"{float(trades['hold_minutes'].median()):.1f} / max {int(trades['hold_minutes'].max())}")
    print(f"entry |ZAP SIG| (sigmas): median {float(trades['entry_dev'].abs().median()):.2f} / "
          f"p90 {float(trades['entry_dev'].abs().quantile(0.9)):.2f}")
    print(f"sides: pos(short) {int((trades['sign'] == 'pos').sum()):,} / "
          f"neg(long) {int((trades['sign'] == 'neg').sum()):,}")
    print(f"mean pnl {float(trades['pnl'].mean()):+.4f} pp, win rate "
          f"{float((trades['pnl'] > 0).mean()):.3f}")

lambda_by_ticker = compute_lambda(entries)
print(f"lambda: {len(lambda_by_ticker):,} tickers")

summary, alpha = build_summary(trades, signals, CFG, sigma_by_ticker, lambda_by_ticker, static_by_ticker)
RATINGS_PATH = OUT_DIR / "opgstretch_ratings.csv"
summary.to_csv(RATINGS_PATH, index=False)

# the scout file - same shape every other strategy's own rolling_perf.json.gz uses
trade_dates_sorted = sorted(trades["trade_date"].unique().tolist()) if len(trades) else []
recent_dates = [str(s) for s in trade_dates_sorted][-255:]
date_index = {int(d): i for i, d in enumerate(recent_dates)}
WINDOWS_DAYS = [5, 20, 40, 65]

CLASS_ORDER = ["EXIT0930"]


def _rate(r):
    return None if pd.isna(r) else round(float(r), 4)


def _lambda_of(tk):
    return None if tk not in lambda_by_ticker.index else round(float(lambda_by_ticker[tk]), 4)


ROLLING_PATH = OUT_DIR / "opgstretch_rolling_perf.json.gz"
by_ticker_ratings: dict = {}
for _, r in summary.iterrows():
    by_ticker_ratings.setdefault(r["ticker"], []).append({
        "cls": r["cls"], "sign": r["sign"], "total": int(r["total"]), "wins": int(r["wins"]),
        "win_rate": _rate(r["win_rate"]), "win_rate_lb": _rate(r["win_rate_lb"]),
        "mean_pnl": _rate(r["mean_pnl"]), "pnl_lb": _rate(r["pnl_lb"]),
        "gamma": _rate(r["gamma"]), "gamma_n": None if pd.isna(r["gamma_n"]) else int(r["gamma_n"]),
        "gamma_rate": _rate(r["gamma_rate"]),
    })


def _alpha_side(tk, col):
    if tk not in alpha.index:
        return None
    v = alpha.loc[tk, col]
    return None if pd.isna(v) else round(float(v), 4)


def _static_of(tk, field):
    """The ticker's IDENTIFICATION row - bench/corr/beta/sigma, the same four on every strategy.
    These are properties of the TICKER, not of this strategy: they come straight off final.parquet,
    where each is constant per ticker. SPECTR renders them and Scanner/Stream/Sonar filter on
    corr/beta, so a null here is a missing column upstream, never a default."""
    v = (static_by_ticker.get(tk) or {}).get(field)
    if v is None:
        return None
    return str(v) if field == "bench" else round(float(v), 4)


rows = []
for tk, g in trades.groupby("ticker") if len(trades) else []:
    rows.append({
        "ticker": tk,
        "alpha_pos": _alpha_side(tk, "alpha_pos"), "alpha_neg": _alpha_side(tk, "alpha_neg"),
        "bench": _static_of(tk, "bench"), "corr": _static_of(tk, "corr"),
        "beta": _static_of(tk, "beta"), "sigma": _static_of(tk, "sigma"),
        "lambda": _lambda_of(tk),
        "ratings": by_ticker_ratings.get(tk, []),
        "trades": [{
            "cls": r["cls"], "sign": r["sign"], "date": date_index.get(int(r["trade_date"])),
            "entry_dev": round(float(r["entry_dev"]), 4), "pnl_pct": round(float(r["pnl"]), 4),
            "status": "win" if r["pnl"] > 0 else "loss",
            "signal_minute_idx": int(r["signal_minute"]), "entry_minute_idx": int(r["entry_minute"]),
            "exit_minute_idx": int(r["exit_minute"]),
        } for _, r in g.iterrows() if int(r["trade_date"]) in date_index],
    })
seen = {r["ticker"] for r in rows}
for tk in alpha.index:
    if tk in seen:
        continue
    ap, an = _alpha_side(tk, "alpha_pos"), _alpha_side(tk, "alpha_neg")
    if ap is None and an is None:
        continue
    rows.append({
        "ticker": tk, "alpha_pos": ap, "alpha_neg": an,
        "bench": _static_of(tk, "bench"), "corr": _static_of(tk, "corr"),
        "beta": _static_of(tk, "beta"), "sigma": _static_of(tk, "sigma"),
        "lambda": _lambda_of(tk),
        "ratings": by_ticker_ratings.get(tk, []), "trades": [],
    })

with gzip.open(ROLLING_PATH, "wt", encoding="utf-8", newline="\n") as f:
    json.dump({
        "meta": {
            "version": "opgstretch-rolling-perf-v1",
            "generated_at": datetime.datetime.utcnow().isoformat() + "Z",
            "most_recent_session": recent_dates[-1] if recent_dates else None,
            "sessions_seen": len(recent_dates),
            "windows_days": WINDOWS_DAYS,
            "recent_dates": recent_dates,
            "position_usd": 1000,
            "min_dev_sig": CFG["min_dev_sig"],
            # SPELLED OUT because this is the one place this family differs in UNITS from every other
            # OPG file, and a consumer that assumes pp would be wrong by a factor of sigma with no
            # symptom other than a gate that never fires.
            "signal_field": "dev_sig - final.parquet's own ZAP SIG column, which IS deviation/sigma "
                            "exactly (verified on 7.3M rows). Read as published rather than "
                            "recomputed from the `deviation` column: CRACEN divides BEFORE rounding, "
                            "so dev_sig carries the full-precision numerator while `deviation` is "
                            "rounded to 2 dp - recomputing is up to 0.281 off, worst where sigma is "
                            "small.",
            "alpha_unit": "sigma",
            "gamma_unit": "sigma",
            "lambda_unit": "pp",
            "pnl_unit": "pp",
            "pnl_basis": "raw Stack% move from the TRIGGERING MINUTE inside 9:15-9:21 to the 9:30 "
                         "print (same calendar morning, same settled prev_close for both readings). "
                         "Entry is the first minute at which |ZAP SIG| > 1 sigma AND a price exists - "
                         "unlike every other OPG family, the entry is NOT the 9:30 open. Positive "
                         "reading -> Short, negative -> Long. One class only.",
            "lambda_basis": "per-ticker root-mean-square of the last pre-close (15:59, back to 15:54) "
                            "Stack% reading, in pp - the identical statistic in every family, and NOT "
                            "this strategy's entry price. Gate-free, same population as alpha.",
            "checkpoints": {k: [v.smin, v.tol] for k, v in CHECKPOINTS.items()},
            "signal_window": [SIGNAL_FROM_MIN, SIGNAL_TO_MIN],
        },
        "rows": rows,
    }, f, ensure_ascii=False)

# Derived from the paths actually written, never a literal: the sibling OPG-fade file inherited this
# line saying "opgreversal_ratings.csv" while the file on disk was correctly named, so the log
# claimed the wrong strategy had been published.
print("written ->", OUT_DIR, "(", RATINGS_PATH.name, ROLLING_PATH.name, ")")
